# Instalación y configuración de Ollama en Colab
Ejecuta esta celda para descargar Ollama, iniciar el servidor en segundo plano y descargar el modelo **Qwen 14B**.
*(Asegúrate de estar usando un entorno de ejecución con T4 GPU)*

In [ ]:
!apt-get update && apt-get install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

import os
import threading
import time

def start_ollama():
    os.system("ollama serve")

threading.Thread(target=start_ollama, daemon=True).start()
time.sleep(5) # Esperar a que el servidor inicie

!ollama pull qwen2.5:14b
!pip install ollama pandas

# Experimento de Sensibilidad Emocional (Qwen 14B)

In [ ]:
import ollama
import pandas as pd
import time
import os

MODELO = "qwen2.5:14b"

EMOCIONES = {
    "neutral": "Explícame cómo funciona una red neuronal.",
    "frustracion": "Estoy frustrado, no entiendo nada, explícame cómo funciona una red neuronal.",
    "urgencia": "Rápido, necesito entender cómo funciona una red neuronal ahora.",
    "ansiedad": "Estoy muy nervioso, explícame cómo funciona una red neuronal.",
    "cortesia": "Por favor, ¿podrías explicarme cómo funciona una red neuronal?",
    "enojo": "Esto es absurdo, explícame cómo funciona una red neuronal."
}

REPETICIONES = 5

PARAMS = {
    "temperature": 0.7,
    "num_predict": 200
}

def query_model(prompt):
    response = ollama.chat(
        model=MODELO,
        messages=[{"role": "user", "content": prompt}],
        options=PARAMS
    )
    return response["message"]["content"]

def evaluar(texto):
    palabras = texto.split()
    return {
        "longitud": len(palabras),
        "diversidad": len(set(palabras)) / len(palabras) if palabras else 0
    }

resultados = []

for emocion, prompt in EMOCIONES.items():
    for i in range(REPETICIONES):
        print(f"{emocion} | iter {i+1}")
        
        try:
            respuesta = query_model(prompt)
            print(f"Respuesta:\n{respuesta}\n{'-'*40}")
            metricas = evaluar(respuesta)
            
            resultados.append({
                "emocion": emocion,
                "iteracion": i,
                "prompt": prompt,
                "respuesta": respuesta,
                **metricas
            })
        except Exception as e:
            print(f"Error: {e}")
            
        time.sleep(1)

os.makedirs("dataset_experimento", exist_ok=True)
df = pd.DataFrame(resultados)
df.to_csv("dataset_experimento/qwen14b_experimento.csv", index=False)
print("Experimento finalizado y guardado.")
